In [6]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from tqdm import tqdm
import os

# تحميل الداتا
data = pd.read_csv('climate_data_with_elevation.csv')

# استخراج النقاط الفريدة (LAT, LON)
unique_points = data[['LAT', 'LON']].drop_duplicates().reset_index(drop=True)
print(f"عدد النقاط الفريدة: {len(unique_points)}")

# تحويل النقاط إلى GeoDataFrame
geometry = [Point(xy) for xy in zip(unique_points['LON'], unique_points['LAT'])]
gdf_points = gpd.GeoDataFrame(unique_points, geometry=geometry, crs="EPSG:4326")

# تحميل بيانات الأنهار من OpenStreetMap
print("جاري تحميل بيانات الأنهار من egypt_rivers_updated.geojson...")
try:
    rivers_osm_gdf = gpd.read_file('egypt_rivers_updated.geojson')
    rivers_osm_gdf = rivers_osm_gdf[rivers_osm_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]
except Exception as e:
    print(f"خطأ أثناء تحميل egypt_rivers_updated.geojson: {e}")
    print("تأكدي إن الملف 'egypt_rivers_updated.geojson' موجود في نفس المجلد بتاع الكود")
    exit()

# التأكد من نظام الإحداثيات
rivers_osm_gdf = rivers_osm_gdf.set_crs(epsg=4326, allow_override=True)

# التحقق من الأشكال الجغرافية الصالحة
rivers_osm_gdf = rivers_osm_gdf[rivers_osm_gdf.geometry.is_valid]

# التحقق من وجود أنهار
print(f"عدد الأنهار/القنوات من OpenStreetMap: {len(rivers_osm_gdf)}")
if rivers_osm_gdf.empty:
    print("لا توجد أنهار في ملف egypt_rivers_updated.geojson. تحققي من الملف أو المصدر.")
    exit()

# تنظيف أسماء الأنهار من OpenStreetMap
if 'name' not in rivers_osm_gdf.columns:
    print("تحذير: عمود 'name' غير موجود في ملف OpenStreetMap.")
    rivers_osm_gdf['name'] = 'Unknown'
else:
    rivers_osm_gdf['name'] = rivers_osm_gdf['name'].fillna('Unknown')

# إضافة عمود 'river_name' بناءً على 'name', 'name:ar', 'name:en', 'destination'
rivers_osm_gdf['river_name'] = rivers_osm_gdf['name']
for col in ['name:ar', 'name:en', 'destination']:
    if col in rivers_osm_gdf.columns:
        rivers_osm_gdf['river_name'] = rivers_osm_gdf.apply(
            lambda row: row[col] if (row['river_name'] == 'Unknown' and pd.notna(row[col])) else row['river_name'],
            axis=1
        )
rivers_osm_gdf['river_name'] = rivers_osm_gdf['river_name'].fillna('Unknown')

# استخراج أجزاء نهر النيل من OpenStreetMap
print("جاري استخراج أجزاء نهر النيل من OpenStreetMap...")
nile_osm_gdf = rivers_osm_gdf[
    rivers_osm_gdf['river_name'].str.contains('Nile|النيل|River Nile', case=False, na=False)
]
# إضافة فحص للأعمدة الأخرى لو 'river_name' ما جابش نتايج كافية
if len(nile_osm_gdf) < 10:
    conditions = []
    if 'name' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['name'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    if 'name:ar' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['name:ar'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    if 'name:en' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['name:en'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    if 'destination' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['destination'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    
    if conditions:
        combined_condition = conditions[0]
        for cond in conditions[1:]:
            combined_condition = combined_condition | cond
        nile_osm_gdf = rivers_osm_gdf[combined_condition]

print(f"عدد الأجزاء المستخرجة لنهر النيل من OpenStreetMap: {len(nile_osm_gdf)}")
if nile_osm_gdf.empty:
    print("تحذير: لم يتم العثور على أجزاء لنهر النيل في ملف OpenStreetMap. هيتم وضع قيم فاضية في عمود distance_to_nile_osm_km.")

# تحميل بيانات الأنهار من Natural Earth
print("جاري تحميل بيانات الأنهار من Natural Earth (ne_10m_rivers_lake_centerlines.shp)...")
try:
    rivers_ne_gdf = gpd.read_file('ne_10m_rivers_lake_centerlines.shp')
    rivers_ne_gdf = rivers_ne_gdf[rivers_ne_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]
except Exception as e:
    print(f"خطأ أثناء تحميل ne_10m_rivers_lake_centerlines.shp: {e}")
    print("تأكدي إن الملف 'ne_10m_rivers_lake_centerlines.shp' وملفاته المرتبطة (زي .dbf) موجودة في نفس المجلد بتاع الكود")
    exit()

# التأكد من نظام الإحداثيات
rivers_ne_gdf = rivers_ne_gdf.set_crs(epsg=4326, allow_override=True)

# التحقق من الأشكال الجغرافية الصالحة
rivers_ne_gdf = rivers_ne_gdf[rivers_ne_gdf.geometry.is_valid]

# التحقق من وجود أنهار
print(f"عدد الأنهار من Natural Earth: {len(rivers_ne_gdf)}")
if rivers_ne_gdf.empty:
    print("لا توجد أنهار في ملف Natural Earth. تحققي من الملف أو المصدر.")
    exit()

# تنظيف أسماء الأنهار من Natural Earth
if 'name' not in rivers_ne_gdf.columns:
    print("تحذير: عمود 'name' غير موجود في ملف Natural Earth.")
    rivers_ne_gdf['name'] = 'Unknown'
else:
    rivers_ne_gdf['name'] = rivers_ne_gdf['name'].fillna('Unknown')

# استخراج أجزاء نهر النيل من Natural Earth
print("جاري استخراج أجزاء نهر النيل من Natural Earth...")
nile_ne_gdf = rivers_ne_gdf[
    rivers_ne_gdf['name'].str.contains('Nile|النيل|River Nile', case=False, na=False)
]
for col in ['name_en', 'name_ar']:
    if col in rivers_ne_gdf.columns:
        temp_nile = rivers_ne_gdf[
            rivers_ne_gdf[col].str.contains('Nile|النيل|River Nile', case=False, na=False)
        ]
        nile_ne_gdf = pd.concat([nile_ne_gdf, temp_nile]).drop_duplicates()

print(f"عدد الأجزاء المستخرجة لنهر النيل من Natural Earth: {len(nile_ne_gdf)}")
if nile_ne_gdf.empty:
    print("تحذير: لم يتم العثور على أجزاء لنهر النيل في ملف Natural Earth. هيتم وضع قيم فاضية في عمود distance_to_nile_ne_km.")

# إنشاء الأعمدة للنتايج
unique_points['distance_to_river_osm_km'] = pd.Series(dtype='float64')
unique_points['nearest_river_osm_name'] = pd.Series(dtype='object')
unique_points['distance_to_nile_osm_km'] = pd.Series(dtype='float64')
unique_points['distance_to_river_ne_km'] = pd.Series(dtype='float64')
unique_points['nearest_river_ne_name'] = pd.Series(dtype='object')
unique_points['distance_to_nile_ne_km'] = pd.Series(dtype='float64')

# دالة لتحديد نظام UTM المناسب بناءً على الإحداثيات
def get_utm_zone(lon):
    zone = int((lon + 180) / 6) + 1
    return f"EPSG:326{zone:02d}"

# دالة لحساب المسافة وجلب اسم أقرب نهر (بالكيلومترات)
def calculate_distance_to_river(point, rivers_gdf, lat, lon, source_name):
    try:
        # تحديد نظام UTM المناسب
        utm_crs = get_utm_zone(lon)
        
        # تحويل النقطة والأنهار إلى نظام UTM لحساب المسافة بدقة (بالمتر)
        point_utm = gpd.GeoSeries([point], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
        rivers_utm = rivers_gdf.to_crs(utm_crs)
        
        # التحقق من rivers_utm
        if rivers_utm.empty:
            print(f"لا توجد أنهار بعد التحويل لـ UTM للنقطة ({lon}, {lat}) من {source_name}")
            return np.nan, 'Unknown'
        
        # التأكد من الأشكال الجغرافية الصالحة بعد التحويل
        rivers_utm = rivers_utm[rivers_utm.geometry.is_valid]
        if rivers_utm.empty:
            print(f"لا توجد أشكال جغرافية صالحة بعد التحويل لـ UTM للنقطة ({lon}, {lat}) من {source_name}")
            return np.nan, 'Unknown'
        
        # حساب المسافة لأقرب نهر (بالمتر)
        distances = rivers_utm.distance(point_utm)
        
        # التحقق من المسافات
        if distances.empty or distances.isna().all():
            print(f"لا توجد مسافات محسوبة للنقطة ({lon}, {lat}) من {source_name}")
            return np.nan, 'Unknown'
        
        min_distance_meters = distances.min()
        nearest_river_idx = distances.idxmin()
        
        # تحويل المسافة من أمتار إلى كيلومترات
        min_distance_km = min_distance_meters / 1000
        
        # طباعة المسافة للتصحيح
        print(f"المسافة لأقرب نهر للنقطة ({lon}, {lat}) من {source_name}: {min_distance_km} كم")
        
        # جلب اسم أقرب نهر
        if source_name == "OpenStreetMap":
            nearest_river_name = rivers_utm.loc[nearest_river_idx, 'river_name']
        else:
            nearest_river_name = rivers_utm.loc[nearest_river_idx, 'name']
        if pd.isna(nearest_river_name) or nearest_river_name == '':
            nearest_river_name = 'Unknown'
        
        return min_distance_km, nearest_river_name
    
    except Exception as e:
        print(f"خطأ عند حساب المسافة للنقطة ({lon}, {lat}) من {source_name}: {e}")
        return np.nan, 'Unknown'

# دالة لحساب المسافة لنهر النيل فقط (بالكيلومترات)
def calculate_distance_to_nile(point, nile_gdf, lat, lon, source_name):
    try:
        if nile_gdf.empty:
            return np.nan
        
        # تحديد نظام UTM المناسب
        utm_crs = get_utm_zone(lon)
        
        # تحويل النقطة وأجزاء النيل إلى نظام UTM لحساب المسافة بدقة (بالمتر)
        point_utm = gpd.GeoSeries([point], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
        nile_utm = nile_gdf.to_crs(utm_crs)
        
        # التحقق من nile_utm
        if nile_utm.empty:
            print(f"لا توجد أجزاء لنهر النيل بعد التحويل لـ UTM للنقطة ({lon}, {lat}) من {source_name}")
            return np.nan
        
        # التأكد من الأشكال الجغرافية الصالحة بعد التحويل
        nile_utm = nile_utm[nile_utm.geometry.is_valid]
        if nile_utm.empty:
            print(f"لا توجد أشكال جغرافية صالحة لنهر النيل بعد التحويل لـ UTM للنقطة ({lon}, {lat}) من {source_name}")
            return np.nan
        
        # حساب المسافة لأقرب جزء من نهر النيل (بالمتر)
        distances = nile_utm.distance(point_utm)
        
        # التحقق من المسافات
        if distances.empty or distances.isna().all():
            print(f"لا توجد مسافات محسوبة لنهر النيل للنقطة ({lon}, {lat}) من {source_name}")
            return np.nan
        
        min_distance_meters = distances.min()
        
        # تحويل المسافة من أمتار إلى كيلومترات
        min_distance_km = min_distance_meters / 1000
        
        # طباعة المسافة للتصحيح
        print(f"المسافة لنهر النيل للنقطة ({lon}, {lat}) من {source_name}: {min_distance_km} كم")
        
        return min_distance_km
    
    except Exception as e:
        print(f"خطأ عند حساب المسافة لنهر النيل للنقطة ({lon}, {lat}) من {source_name}: {e}")
        return np.nan

# حساب المسافات لكل النقاط
print(f"حساب المسافات لـ {len(unique_points)} نقاط...")
distances_osm = []
river_names_osm = []
distances_nile_osm = []
distances_ne = []
river_names_ne = []
distances_nile_ne = []

for index, row in tqdm(unique_points.iterrows(), total=len(unique_points), desc="حساب المسافات"):
    # حساب المسافة لأقرب نهر من OpenStreetMap
    distance_osm, river_name_osm = calculate_distance_to_river(
        gdf_points.loc[index, 'geometry'], 
        rivers_osm_gdf, 
        row['LAT'], 
        row['LON'],
        "OpenStreetMap"
    )
    distances_osm.append(distance_osm)
    river_names_osm.append(river_name_osm)
    
    # حساب المسافة لنهر النيل من OpenStreetMap
    distance_nile_osm = calculate_distance_to_nile(
        gdf_points.loc[index, 'geometry'], 
        nile_osm_gdf, 
        row['LAT'], 
        row['LON'],
        "OpenStreetMap"
    )
    distances_nile_osm.append(distance_nile_osm)
    
    # حساب المسافة لأقرب نهر من Natural Earth
    distance_ne, river_name_ne = calculate_distance_to_river(
        gdf_points.loc[index, 'geometry'], 
        rivers_ne_gdf, 
        row['LAT'], 
        row['LON'],
        "Natural Earth"
    )
    distances_ne.append(distance_ne)
    river_names_ne.append(river_name_ne)
    
    # حساب المسافة لنهر النيل من Natural Earth
    distance_nile_ne = calculate_distance_to_nile(
        gdf_points.loc[index, 'geometry'], 
        nile_ne_gdf, 
        row['LAT'], 
        row['LON'],
        "Natural Earth"
    )
    distances_nile_ne.append(distance_nile_ne)

# تحديث الأعمدة في unique_points
unique_points['distance_to_river_osm_km'] = distances_osm
unique_points['nearest_river_osm_name'] = river_names_osm
unique_points['distance_to_nile_osm_km'] = distances_nile_osm
unique_points['distance_to_river_ne_km'] = distances_ne
unique_points['nearest_river_ne_name'] = river_names_ne
unique_points['distance_to_nile_ne_km'] = distances_nile_ne

# التحقق من النتايج
print("عينة من النتايج:")
print(unique_points[['LAT', 'LON', 'distance_to_river_osm_km', 'nearest_river_osm_name', 'distance_to_nile_osm_km',
                    'distance_to_river_ne_km', 'nearest_river_ne_name', 'distance_to_nile_ne_km']].head(10))

# التحقق من القيم الفاضية
for column in ['distance_to_river_osm_km', 'nearest_river_osm_name', 'distance_to_nile_osm_km',
               'distance_to_river_ne_km', 'nearest_river_ne_name', 'distance_to_nile_ne_km']:
    if unique_points[column].isna().any():
        print(f"تحذير: يوجد قيم فاضية في عمود {column}")
        print(f"عدد القيم الفاضية: {unique_points[column].isna().sum()}")

# حفظ النتايج في ملف منفصل
output_file = 'river_distances_separate.csv'
unique_points[['LAT', 'LON', 'distance_to_river_osm_km', 'nearest_river_osm_name', 'distance_to_nile_osm_km',
               'distance_to_river_ne_km', 'nearest_river_ne_name', 'distance_to_nile_ne_km']].to_csv(output_file, index=False)
print(f"تم حفظ النتايج في ملف منفصل: {output_file}")

عدد النقاط الفريدة: 90
جاري تحميل بيانات الأنهار من egypt_rivers_updated.geojson...
عدد الأنهار/القنوات من OpenStreetMap: 8607
جاري استخراج أجزاء نهر النيل من OpenStreetMap...
عدد الأجزاء المستخرجة لنهر النيل من OpenStreetMap: 71
جاري تحميل بيانات الأنهار من Natural Earth (ne_10m_rivers_lake_centerlines.shp)...
عدد الأنهار من Natural Earth: 1473
جاري استخراج أجزاء نهر النيل من Natural Earth...
عدد الأجزاء المستخرجة لنهر النيل من Natural Earth: 13
حساب المسافات لـ 90 نقاط...


حساب المسافات:   0%|          | 0/90 [00:00<?, ?it/s]

المسافة لأقرب نهر للنقطة (25.5, 22.5) من OpenStreetMap: 61.12943432744669 كم
المسافة لنهر النيل للنقطة (25.5, 22.5) من OpenStreetMap: 533.8819044346302 كم


حساب المسافات:   1%|          | 1/90 [00:00<00:53,  1.68it/s]

المسافة لأقرب نهر للنقطة (25.5, 22.5) من Natural Earth: 535.6951613686256 كم
المسافة لنهر النيل للنقطة (25.5, 22.5) من Natural Earth: 535.6951613686256 كم


...

المسافة لأقرب نهر للنقطة (30.5, 31.5) من OpenStreetMap: 7.783203579176016 كم
المسافة لنهر النيل للنقطة (30.5, 31.5) من OpenStreetMap: 10.15815277573869 كم
المسافة لأقرب نهر للنقطة (30.5, 31.5) من Natural Earth: 10.399319673850163 كم
المسافة لنهر النيل للنقطة (30.5, 31.5) من Natural Earth: 168.09485984432226 كم


حساب المسافات: 100%|██████████| 90/90 [00:49<00:00,  1.76it/s]

المسافة لأقرب نهر للنقطة (31.5, 31.5) من OpenStreetMap: 4.287973765485492 كم
المسافة لنهر النيل للنقطة (31.5, 31.5) من OpenStreetMap: 24.320984131255283 كم
المسافة لأقرب نهر للنقطة (31.5, 31.5) من Natural Earth: 24.427705560153953 كم
المسافة لنهر النيل للنقطة (31.5, 31.5) من Natural Earth: 154.61662631586 كم


حساب المسافات: 100%|██████████| 90/90 [00:49<00:00,  1.82it/s]

عينة من النتايج:
    LAT   LON  distance_to_river_osm_km        nearest_river_osm_name  \
0  22.5  25.5                 61.129434                     وادي صورة   
1  22.5  26.5                 83.076458  وادى المفتوح Wadi El Maftouh   
2  22.5  27.5                129.498711  وادى المفتوح Wadi El Maftouh   
3  22.5  28.5                195.926776                       Unknown   
4  22.5  29.5                113.158071                       Unknown   
5  22.5  30.5                 32.399502                       Unknown   
6  22.5  31.5                 14.205224                       Unknown   
7  22.5  32.5                  8.530057                       Kurusku   
8  22.5  33.5                  4.809108                   Wadi Allaqi   
9  22.5  34.5                 45.222342                   Wadi Allaqi   

   distance_to_nile_osm_km  distance_to_river_ne_km nearest_river_ne_name  \
0               533.881904               535.695161                  Nile   
1               438.81022

In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from tqdm import tqdm

# 1. قراءة ملف climate_data_with_elevation.csv
data = pd.read_csv('climate_data_with_elevation.csv')

# 2. استخراج النقاط الفريدة (LAT, LON)
unique_points = data[['LAT', 'LON']].drop_duplicates().reset_index(drop=True)
print(f"عدد النقاط الفريدة: {len(unique_points)}")

# 3. تحويل النقاط إلى GeoDataFrame
geometry = [Point(xy) for xy in zip(unique_points['LON'], unique_points['LAT'])]
gdf_points = gpd.GeoDataFrame(unique_points, geometry=geometry, crs="EPSG:4326")

# 4. قراءة ملف egypt_rivers.geojson
print("جاري تحميل بيانات الأنهار من egypt_rivers.geojson...")
try:
    rivers_osm_gdf = gpd.read_file('egypt_rivers.geojson')
    rivers_osm_gdf = rivers_osm_gdf[rivers_osm_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]
except Exception as e:
    print(f"خطأ أثناء تحميل egypt_rivers.geojson: {e}")
    print("تأكدي إن الملف 'egypt_rivers.geojson' موجود في نفس المجلد بتاع الكود")
    exit()

# 5. التأكد من نظام الإحداثيات
rivers_osm_gdf = rivers_osm_gdf.set_crs(epsg=4326, allow_override=True)

# 6. التحقق من الأشكال الجغرافية الصالحة
rivers_osm_gdf = rivers_osm_gdf[rivers_osm_gdf.geometry.is_valid]

# 7. التحقق من وجود أنهار
print(f"عدد الأنهار/القنوات من egypt_rivers.geojson: {len(rivers_osm_gdf)}")
if rivers_osm_gdf.empty:
    print("لا توجد أنهار في ملف egypt_rivers.geojson. تحققي من الملف أو المصدر.")
    exit()

# 8. تنظيف أسماء الأنهار
if 'name' not in rivers_osm_gdf.columns:
    print("تحذير: عمود 'name' غير موجود في ملف egypt_rivers.geojson.")
    rivers_osm_gdf['name'] = 'Unknown'
else:
    rivers_osm_gdf['name'] = rivers_osm_gdf['name'].fillna('Unknown')

# إضافة عمود 'river_name' بناءً على 'name', 'name:ar', 'name:en', 'destination'
rivers_osm_gdf['river_name'] = rivers_osm_gdf['name']
for col in ['name:ar', 'name:en', 'destination']:
    if col in rivers_osm_gdf.columns:
        rivers_osm_gdf['river_name'] = rivers_osm_gdf.apply(
            lambda row: row[col] if (row['river_name'] == 'Unknown' and pd.notna(row[col])) else row['river_name'],
            axis=1
        )
rivers_osm_gdf['river_name'] = rivers_osm_gdf['river_name'].fillna('Unknown')

# 9. استخراج أجزاء نهر النيل
print("جاري استخراج أجزاء نهر النيل من egypt_rivers.geojson...")
nile_osm_gdf = rivers_osm_gdf[
    rivers_osm_gdf['river_name'].str.contains('Nile|النيل|River Nile', case=False, na=False)
]

# إضافة فحص للأعمدة الأخرى لو 'river_name' ما جابش نتايج كافية
if len(nile_osm_gdf) < 1:  # لو ما لقيناش النيل
    conditions = []
    if 'name' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['name'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    if 'name:ar' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['name:ar'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    if 'name:en' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['name:en'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    if 'destination' in rivers_osm_gdf.columns:
        conditions.append(rivers_osm_gdf['destination'].str.contains('Nile|النيل|River Nile', case=False, na=False))
    
    if conditions:
        combined_condition = conditions[0]
        for cond in conditions[1:]:
            combined_condition = combined_condition | cond
        nile_osm_gdf = rivers_osm_gdf[combined_condition]

print(f"عدد الأجزاء المستخرجة لنهر النيل من egypt_rivers.geojson: {len(nile_osm_gdf)}")
if nile_osm_gdf.empty:
    print("خطأ: لم يتم العثور على أجزاء لنهر النيل في ملف egypt_rivers.geojson. تحققي من الملف.")
    exit()

# 10. إنشاء عمود للمسافة لنهر النيل
unique_points['Distance_to_Nile_km'] = pd.Series(dtype='float64')

# 11. دالة لتحديد نظام UTM المناسب بناءً على الإحداثيات
def get_utm_zone(lon):
    zone = int((lon + 180) / 6) + 1
    return f"EPSG:326{zone:02d}"

# 12. دالة لحساب المسافة لنهر النيل (بالكيلومترات)
def calculate_distance_to_nile(point, nile_gdf, lat, lon):
    try:
        if nile_gdf.empty:
            return np.nan
        
        # تحديد نظام UTM المناسب
        utm_crs = get_utm_zone(lon)
        
        # تحويل النقطة وأجزاء النيل إلى نظام UTM لحساب المسافة بدقة (بالمتر)
        point_utm = gpd.GeoSeries([point], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
        nile_utm = nile_gdf.to_crs(utm_crs)
        
        # التحقق من nile_utm
        if nile_utm.empty:
            print(f"لا توجد أجزاء لنهر النيل بعد التحويل لـ UTM للنقطة ({lon}, {lat})")
            return np.nan
        
        # التأكد من الأشكال الجغرافية الصالحة بعد التحويل
        nile_utm = nile_utm[nile_utm.geometry.is_valid]
        if nile_utm.empty:
            print(f"لا توجد أشكال جغرافية صالحة لنهر النيل بعد التحويل لـ UTM للنقطة ({lon}, {lat})")
            return np.nan
        
        # حساب المسافة لأقرب جزء من نهر النيل (بالمتر)
        distances = nile_utm.distance(point_utm)
        
        # التحقق من المسافات
        if distances.empty or distances.isna().all():
            print(f"لا توجد مسافات محسوبة لنهر النيل للنقطة ({lon}, {lat})")
            return np.nan
        
        min_distance_meters = distances.min()
        
        # تحويل المسافة من أمتار إلى كيلومترات
        min_distance_km = min_distance_meters / 1000
        
        # طباعة المسافة للتصحيح
        print(f"المسافة لنهر النيل للنقطة ({lon}, {lat}): {min_distance_km} كم")
        
        return min_distance_km
    
    except Exception as e:
        print(f"خطأ عند حساب المسافة لنهر النيل للنقطة ({lon}, {lat}): {e}")
        return np.nan

# 13. حساب المسافات لكل النقاط
print(f"حساب المسافات لـ {len(unique_points)} نقاط...")
distances_nile = []

for index, row in tqdm(unique_points.iterrows(), total=len(unique_points), desc="حساب المسافات"):
    # حساب المسافة لنهر النيل
    distance_nile = calculate_distance_to_nile(
        gdf_points.loc[index, 'geometry'], 
        nile_osm_gdf, 
        row['LAT'], 
        row['LON']
    )
    distances_nile.append(distance_nile)

# تحديث العمود في unique_points
unique_points['Distance_to_Nile_km'] = distances_nile

# 14. التحقق من النتايج
print("عينة من النتايج:")
print(unique_points[['LAT', 'LON', 'Distance_to_Nile_km']].head(10))

# التحقق من القيم الفاضية
if unique_points['Distance_to_Nile_km'].isna().any():
    print(f"تحذير: يوجد قيم فاضية في عمود Distance_to_Nile_km")
    print(f"عدد القيم الفاضية: {unique_points['Distance_to_Nile_km'].isna().sum()}")

# 15. حفظ النقاط الفريدة مع المسافات في ملف منفصل
unique_points[['LAT', 'LON', 'Distance_to_Nile_km']].to_csv("unique_points_with_nile_distance.csv", index=False)
print("تم حفظ النقاط الفريدة مع المسافات في ملف: unique_points_with_nile_distance.csv")

# 16. دمج المسافات مع البيانات الأصلية مع التكرارات
data_with_distance = data.merge(
    unique_points[['LAT', 'LON', 'Distance_to_Nile_km']],
    on=['LAT', 'LON'],
    how='left'
)

# 17. حفظ البيانات الأصلية المعدلة مع التكرارات
data_with_distance.to_csv("climate_data_with_elevation.csv", index=False)
print("تم تحديث الملف الأصلي climate_data_with_elevation.csv مع المسافات")

# 18. طباعة أول 5 صفوف للتأكد
print(data_with_distance.head())

عدد النقاط الفريدة: 90
جاري تحميل بيانات الأنهار من egypt_rivers.geojson...
عدد الأنهار/القنوات من egypt_rivers.geojson: 8606
جاري استخراج أجزاء نهر النيل من egypt_rivers.geojson...
عدد الأجزاء المستخرجة لنهر النيل من egypt_rivers.geojson: 70
حساب المسافات لـ 90 نقاط...


حساب المسافات:  21%|██        | 19/90 [00:00<00:00, 90.89it/s]

المسافة لنهر النيل للنقطة (25.5, 22.5): 533.8819044346302 كم
المسافة لنهر النيل للنقطة (26.5, 22.5): 438.8102217954129 كم
المسافة لنهر النيل للنقطة (27.5, 22.5): 348.3191510706053 كم
المسافة لنهر النيل للنقطة (28.5, 22.5): 266.44826898232117 كم
المسافة لنهر النيل للنقطة (29.5, 22.5): 187.1264906911073 كم
المسافة لنهر النيل للنقطة (30.5, 22.5): 101.25415308445689 كم
المسافة لنهر النيل للنقطة (31.5, 22.5): 22.634657284906435 كم
المسافة لنهر النيل للنقطة (32.5, 22.5): 17.11861705633435 كم
المسافة لنهر النيل للنقطة (33.5, 22.5): 99.65629769973921 كم
المسافة لنهر النيل للنقطة (34.5, 22.5): 184.6109259696284 كم
المسافة لنهر النيل للنقطة (35.5, 22.5): 278.47749228695244 كم
المسافة لنهر النيل للنقطة (25.5, 23.5): 581.02029751497 كم
المسافة لنهر النيل للنقطة (26.5, 23.5): 495.47487205309545 كم
المسافة لنهر النيل للنقطة (27.5, 23.5): 413.95658795798937 كم
المسافة لنهر النيل للنقطة (28.5, 23.5): 333.90149897148035 كم
المسافة لنهر النيل للنقطة (29.5, 23.5): 248.0489937052358 كم
المسافة لنهر النيل 

حساب المسافات:  34%|███▍      | 31/90 [00:00<00:00, 103.32it/s]

المسافة لنهر النيل للنقطة (29.5, 24.5): 314.92895650254076 كم
المسافة لنهر النيل للنقطة (30.5, 24.5): 225.71338386764702 كم
المسافة لنهر النيل للنقطة (31.5, 24.5): 137.67772303612756 كم
المسافة لنهر النيل للنقطة (32.5, 24.5): 37.49553596176846 كم
المسافة لنهر النيل للنقطة (33.5, 24.5): 58.71890540904397 كم
المسافة لنهر النيل للنقطة (34.5, 24.5): 159.6481725165591 كم
المسافة لنهر النيل للنقطة (25.5, 25.5): 575.6235543724916 كم
المسافة لنهر النيل للنقطة (26.5, 25.5): 484.7769389069421 كم
المسافة لنهر النيل للنقطة (27.5, 25.5): 397.85652780587003 كم
المسافة لنهر النيل للنقطة (28.5, 25.5): 318.1794160918812 كم
المسافة لنهر النيل للنقطة (29.5, 25.5): 247.6211821763578 كم
المسافة لنهر النيل للنقطة (30.5, 25.5): 167.1942047652954 كم


...

المسافة لنهر النيل للنقطة (31.5, 25.5): 91.35423401686796 كم
المسافة لنهر النيل للنقطة (32.5, 25.5): 0.7273328959860001 كم
المسافة لنهر النيل للنقطة (33.5, 25.5): 80.2417695211097 كم
المسافة لنهر النيل للنقطة (34.5, 25.5): 172.01607815052932 كم
المسافة لنهر النيل للنقطة (25.5, 26.5): 541.4016463137023 كم
المسافة لنهر النيل للنقطة (26.5, 26.5): 444.52820948283954 كم
المسافة لنهر النيل للنقطة (27.5, 26.5): 348.9851410644893 كم
المسافة لنهر النيل للنقطة (28.5, 26.5): 256.2406631738536 كم
المسافة لنهر النيل للنقطة (29.5, 26.5): 170.17475158915275 كم
المسافة لنهر النيل للنقطة (30.5, 26.5): 100.00638624810266 كم
المسافة لنهر النيل للنقطة (31.5, 26.5): 20.35425335248438 كم
المسافة لنهر النيل للنقطة (32.5, 26.5): 37.93916045883172 كم
المسافة لنهر النيل للنقطة (33.5, 26.5): 85.74212297121714 كم
المسافة لنهر النيل للنقطة (25.5, 27.5): 521.3550784192522 كم
المسافة لنهر النيل للنقطة (26.5, 27.5): 424.0654694412833 كم
المسافة لنهر النيل للنقطة (27.5, 27.5): 327.46268394089293 كم


حساب المسافات:  79%|███████▉  | 71/90 [00:00<00:00, 85.14it/s]

المسافة لنهر النيل للنقطة (28.5, 27.5): 231.8574295887621 كم
المسافة لنهر النيل للنقطة (29.5, 27.5): 133.10873093389756 كم
المسافة لنهر النيل للنقطة (30.5, 27.5): 34.53704013714714 كم
المسافة لنهر النيل للنقطة (31.5, 27.5): 42.85129006864345 كم
المسافة لنهر النيل للنقطة (32.5, 27.5): 120.23164599844212 كم
المسافة لنهر النيل للنقطة (25.5, 28.5): 513.8311830374284 كم
المسافة لنهر النيل للنقطة (26.5, 28.5): 416.00250318428976 كم
المسافة لنهر النيل للنقطة (27.5, 28.5): 318.24859245120507 كم
المسافة لنهر النيل للنقطة (28.5, 28.5): 220.64947136089611 كم
المسافة لنهر النيل للنقطة (29.5, 28.5): 123.54434102741571 كم
المسافة لنهر النيل للنقطة (30.5, 28.5): 30.07476442038336 كم
المسافة لنهر النيل للنقطة (31.5, 28.5): 63.68891414691518 كم
المسافة لنهر النيل للنقطة (32.5, 28.5): 148.2204350539655 كم
المسافة لنهر النيل للنقطة (33.5, 28.5): 236.70115007351674 كم
المسافة لنهر النيل للنقطة (25.5, 29.5): 527.9595613923993 كم
المسافة لنهر النيل للنقطة (26.5, 29.5): 433.68202978679136 كم
المسافة لنهر الن

حساب المسافات: 100%|██████████| 90/90 [00:01<00:00, 85.23it/s]


المسافة لنهر النيل للنقطة (26.5, 30.5): 447.90138151680566 كم
المسافة لنهر النيل للنقطة (27.5, 30.5): 352.2015888188043 كم
المسافة لنهر النيل للنقطة (28.5, 30.5): 256.78830677430574 كم
المسافة لنهر النيل للنقطة (29.5, 30.5): 162.18026608717867 كم
المسافة لنهر النيل للنقطة (30.5, 30.5): 71.69621405381773 كم
المسافة لنهر النيل للنقطة (31.5, 30.5): 48.764887349241135 كم
المسافة لنهر النيل للنقطة (32.5, 30.5): 128.48157870841652 كم
المسافة لنهر النيل للنقطة (33.5, 30.5): 221.52057233488256 كم
المسافة لنهر النيل للنقطة (34.5, 30.5): 316.4194060100112 كم
المسافة لنهر النيل للنقطة (25.5, 31.5): 559.4710398965398 كم
المسافة لنهر النيل للنقطة (26.5, 31.5): 467.9164227508448 كم
المسافة لنهر النيل للنقطة (30.5, 31.5): 159.63135668213496 كم
المسافة لنهر النيل للنقطة (31.5, 31.5): 151.24277484066616 كم
عينة من النتايج:
    LAT   LON  Distance_to_Nile_km
0  22.5  25.5           533.881904
1  22.5  26.5           438.810222
2  22.5  27.5           348.319151
3  22.5  28.5           266.448269
4  22.5